# 01 Data Exploration: Marketplace Product Analytics

**Business question**

Users are visiting our marketplace, but not enough are purchasing and returning. Where are we losing users, what behaviors predict retention, and what should Product change?

**Role of this notebook**

This notebook is the first pass EDA. The goal is to understand the dataset, validate key fields, define the event grain, and create early product hypotheses for the deeper notebooks:

- `02_behavioral_eda.ipynb`: user, session, product, brand, and category behavior
- `03_funnel_analysis.ipynb`: view to cart to purchase conversion and drop-off
- `04_retention_analysis.ipynb`: repeat visits, repeat purchases, cohorts, and return behavior
- `05_retention_modeling.ipynb`: features that predict retention


## Product Analytics Readout

Use this section as the executive summary after running the notebook.

**Current read**

- Data grain: one row is one product event by one user in one session at one timestamp.
- Core event types expected: `view`, `cart`, `remove_from_cart`, `purchase`.
- Primary marketplace health metrics: visitors, sessions, product views, carts, purchases, buyer conversion, approximate GMV, and repeat behavior.
- Known caveat: multiple purchase rows in the same session may represent multiple products in one order, not separate orders.

**Open questions for Product**

- Are users dropping before cart, after cart, or after first purchase?
- Which behaviors indicate high intent: carting, repeated product views, category breadth, price band, brand affinity, or session frequency?
- Are missing taxonomy fields concentrated in products that matter for conversion or retention?


## Setup

Keep setup short and reproducible. This notebook samples from the monthly raw files so it runs quickly during exploration. For final metrics, rerun key analyses on the full dataset or use chunked aggregation.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:.2f}".format)

DATA_DIR = Path("../data/raw")
RAW_FILES = sorted(DATA_DIR.glob("2019-*.csv"))
NROWS_PER_FILE = 100_000

RAW_FILES


[PosixPath('../data/raw/2019-Nov.csv'), PosixPath('../data/raw/2019-Oct.csv')]

## Load Data

For first-pass EDA, load the same number of rows from each monthly file. This avoids making the notebook depend only on one month, while still keeping iteration fast.


In [2]:
frames = []

for file_path in RAW_FILES:
    month_df = pd.read_csv(file_path, nrows=NROWS_PER_FILE)
    month_df["source_file"] = file_path.name
    frames.append(month_df)

df_raw = pd.concat(frames, ignore_index=True)

df_raw.head()


,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,source_file
0,2019-11-01 00:00:00 UTC,view,1003461,2053013555631882655,electronics.smartphone,xiaomi,489.07,520088904,4d3b30da-a5e4-49df-b1a8-ba5943f1dd33,2019-Nov.csv
1,2019-11-01 00:00:00 UTC,view,5000088,2053013566100866035,appliances.sewing_machine,janome,293.65,530496790,8e5f4f83-366c-4f70-860e-ca7417414283,2019-Nov.csv
2,2019-11-01 00:00:01 UTC,view,17302664,2053013553853497655,NaN,creed,28.31,561587266,755422e7-9040-477b-9bd2-6a6e8fd97387,2019-Nov.csv
3,2019-11-01 00:00:01 UTC,view,3601530,2053013563810775923,appliances.kitchen.washer,lg,712.87,518085591,3bfb58cd-7892-48cc-8020-2f17e6de6e7f,2019-Nov.csv
4,2019-11-01 00:00:01 UTC,view,1004775,2053013555631882655,electronics.smartphone,xiaomi,183.27,558856683,313628f1-68b8-460d-84f6-cec7a8796ef2,2019-Nov.csv


In [3]:
print(f"Rows loaded: {len(df_raw):,}")
print(f"Columns: {df_raw.shape[1]}")
print(f"Files: {', '.join(df_raw['source_file'].unique())}")


Rows loaded: 200,000
Columns: 10
Files: 2019-Nov.csv, 2019-Oct.csv


## Data Grain And Schema

Expected grain: one event for one product, performed by one user, within one user session, at one timestamp.

A product analyst should confirm this before building funnel or retention metrics, because event-level data can overcount users, sessions, carts, or orders if the grain is misunderstood.


In [4]:
expected_columns = [
    "event_time",
    "event_type",
    "product_id",
    "category_id",
    "category_code",
    "brand",
    "price",
    "user_id",
    "user_session",
]

missing_columns = sorted(set(expected_columns) - set(df_raw.columns))
extra_columns = sorted(set(df_raw.columns) - set(expected_columns) - {"source_file"})

print("Missing expected columns:", missing_columns)
print("Unexpected columns:", extra_columns)

df_raw.dtypes


Missing expected columns: []
Unexpected columns: []


event_time           str
event_type           str
product_id         int64
category_id        int64
category_code        str
brand                str
price            float64
user_id            int64
user_session         str
source_file          str
dtype: object

In [5]:
df = df_raw.copy()

# Convert fields to analytical types.
df["event_time"] = pd.to_datetime(df["event_time"], utc=True)

id_columns = ["product_id", "category_id", "user_id", "user_session"]
df[id_columns] = df[id_columns].astype("string")

df["event_type"] = df["event_type"].astype("category")
df["category_code"] = df["category_code"].astype("string")
df["brand"] = df["brand"].astype("string")

df.dtypes


event_time       datetime64[us, UTC]
event_type                  category
product_id                    string
category_id                   string
category_code                 string
brand                         string
price                        float64
user_id                       string
user_session                  string
source_file                      str
dtype: object

## Data Quality Checks

These checks answer: Can we trust the fields needed for funnel, retention, and product behavior analysis?


In [6]:
quality_summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "rows": len(df),
    "missing": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "unique": df.nunique(dropna=True),
}).sort_values("missing_pct", ascending=False)

quality_summary


,dtype,rows,missing,missing_pct,unique
category_code,string,200000,66427,33.21,126
brand,string,200000,30169,15.08,1990
event_time,"datetime64[us, UTC]",200000,0,0.00,21678
event_type,category,200000,0,0.00,3
product_id,string,200000,0,0.00,33501
category_id,string,200000,0,0.00,589
price,float64,200000,0,0.00,12542
user_id,string,200000,0,0.00,40822
user_session,string,200000,0,0.00,49271
source_file,str,200000,0,0.00,2


In [7]:
duplicate_rows = df.duplicated().sum()
print(f"Duplicate rows: {duplicate_rows:,}")

if duplicate_rows > 0:
    display(
        df.loc[df.duplicated(keep=False)]
          .sort_values(["user_id", "user_session", "event_time"])
          .head(20)
    )


Duplicate rows: 44


,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session,source_file
194134,2019-10-01 04:22:18+00:00,cart,5301617,2053013563173241677,<NA>,polaris,25.17,512469652,dd3a3785-b727-4636-b819-6738ce186d83,2019-Oct.csv
194135,2019-10-01 04:22:18+00:00,cart,5301617,2053013563173241677,<NA>,polaris,25.17,512469652,dd3a3785-b727-4636-b819-6738ce186d83,2019-Oct.csv
16136,2019-11-01 01:36:58+00:00,view,1004173,2053013555631882655,electronics.smartphone,xiaomi,164.36,512595904,85f11e30-0d57-452d-aadc-bd110497aa7c,2019-Nov.csv
16138,2019-11-01 01:36:58+00:00,view,1004173,2053013555631882655,electronics.smartphone,xiaomi,164.36,512595904,85f11e30-0d57-452d-aadc-bd110497aa7c,2019-Nov.csv
168168,2019-10-01 03:54:13+00:00,cart,4804056,2053013554658804075,electronics.audio.headphone,apple,161.93,512759443,06f51dae-879e-4cb8-a201-a1d8e45ff77c,2019-Oct.csv
168170,2019-10-01 03:54:13+00:00,cart,4804056,2053013554658804075,electronics.audio.headphone,apple,161.93,512759443,06f51dae-879e-4cb8-a201-a1d8e45ff77c,2019-Oct.csv
99539,2019-11-01 03:52:17+00:00,view,7600143,2053013552821698803,<NA>,tenda,50.12,514677746,08a0f1dc-17bd-479e-b977-c237f7b7a879,2019-Nov.csv
99541,2019-11-01 03:52:17+00:00,view,7600143,2053013552821698803,<NA>,tenda,50.12,514677746,08a0f1dc-17bd-479e-b977-c237f7b7a879,2019-Nov.csv
44967,2019-11-01 02:39:56+00:00,cart,30000048,2127425436764865054,construction.tools.welding,<NA>,118.90,515136785,d03d2aa1-c23f-4b0d-bdf9-4d9dffc0407d,2019-Nov.csv
44968,2019-11-01 02:39:56+00:00,cart,30000048,2127425436764865054,construction.tools.welding,<NA>,118.90,515136785,d03d2aa1-c23f-4b0d-bdf9-4d9dffc0407d,2019-Nov.csv


In [8]:
time_coverage = pd.Series({
    "min_event_time": df["event_time"].min(),
    "max_event_time": df["event_time"].max(),
    "unique_event_dates": df["event_time"].dt.date.nunique(),
    "source_files": df["source_file"].nunique(),
})

time_coverage


min_event_time        2019-10-01 00:00:00+00:00
max_event_time        2019-11-01 03:52:45+00:00
unique_event_dates                            2
source_files                                  2
dtype: object

## Category Taxonomy QA

`category_code` is useful for interpretable product/category analysis, but it can be missing. Before deciding how to handle it, check whether a `category_id` with missing taxonomy ever has taxonomy populated in another row.


In [9]:
category_code_by_id = (
    df.assign(category_code_missing=df["category_code"].isna() | df["category_code"].str.strip().eq(""))
      .groupby("category_id", dropna=False)
      .agg(
          rows=("category_id", "size"),
          missing_code_rows=("category_code_missing", "sum"),
          known_code_rows=("category_code_missing", lambda s: (~s).sum()),
          unique_known_codes=("category_code", lambda s: s.dropna().nunique()),
          example_category_code=("category_code", lambda s: s.dropna().iloc[0] if s.dropna().size else pd.NA),
      )
      .reset_index()
)

category_ids_missing_code = category_code_by_id.query("missing_code_rows > 0")
category_ids_missing_and_known = category_code_by_id.query("missing_code_rows > 0 and known_code_rows > 0")

print(f"Category IDs with any missing category_code: {len(category_ids_missing_code):,}")
print(f"Of those, category IDs that also have category_code elsewhere: {len(category_ids_missing_and_known):,}")

category_ids_missing_and_known.sort_values("rows", ascending=False).head(20)


Category IDs with any missing category_code: 353
Of those, category IDs that also have category_code elsewhere: 0


,category_id,rows,missing_code_rows,known_code_rows,unique_known_codes,example_category_code


## Marketplace Health Snapshot

This is the standard top-level product analytics view: traffic, engagement, conversion, buyer base, and approximate revenue. It is not the final funnel analysis yet, but it tells us whether the data behaves like a marketplace event log.


In [10]:
purchase_events = df[df["event_type"] == "purchase"]

health_snapshot = pd.Series({
    "events": len(df),
    "users": df["user_id"].nunique(),
    "sessions": df["user_session"].nunique(),
    "products": df["product_id"].nunique(),
    "categories": df["category_id"].nunique(),
    "brands": df["brand"].nunique(),
    "purchase_events": len(purchase_events),
    "purchasing_users": purchase_events["user_id"].nunique(),
    "buyer_conversion_pct": purchase_events["user_id"].nunique() / df["user_id"].nunique() * 100,
    "approx_gmv": purchase_events["price"].sum(),
})

health_snapshot.to_frame("value")


,value
events,200000.00
users,40822.00
sessions,49271.00
products,33501.00
categories,589.00
brands,1990.00
purchase_events,3077.00
purchasing_users,2484.00
buyer_conversion_pct,6.08
approx_gmv,902803.81


In [11]:
event_mix = (
    df["event_type"]
      .value_counts(dropna=False)
      .rename_axis("event_type")
      .reset_index(name="events")
)
event_mix["event_pct"] = (event_mix["events"] / event_mix["events"].sum() * 100).round(2)

event_mix


,event_type,events,event_pct
0,view,194619,97.31
1,purchase,3077,1.54
2,cart,2304,1.15


In [12]:
event_mix_display = event_mix.copy()
event_mix_display["cumulative_event_pct"] = event_mix_display["event_pct"].cumsum().round(2)
event_mix_display


,event_type,events,event_pct,cumulative_event_pct
0,view,194619,97.31,97.31
1,purchase,3077,1.54,98.85
2,cart,2304,1.15,100.00


## First-Pass Session Funnel

For marketplace funnel analysis, session-level conversion is often more useful than event counts because it shows whether a visit contained each behavior at least once.

This is a first pass only. The dedicated funnel notebook should later segment by category, brand, price band, device/source if available, and new vs returning users.


In [13]:
session_funnel = (
    df.pivot_table(
        index="user_session",
        columns="event_type",
        values="event_time",
        aggfunc="size",
        fill_value=0,
        observed=False,
    )
    .reset_index()
)

for event in ["view", "cart", "remove_from_cart", "purchase"]:
    if event not in session_funnel.columns:
        session_funnel[event] = 0

session_flags = session_funnel.assign(
    viewed=session_funnel["view"] > 0,
    carted=session_funnel["cart"] > 0,
    purchased=session_funnel["purchase"] > 0,
)

funnel_summary = pd.Series({
    "sessions": len(session_flags),
    "view_sessions": session_flags["viewed"].sum(),
    "cart_sessions": session_flags["carted"].sum(),
    "purchase_sessions": session_flags["purchased"].sum(),
    "view_to_cart_pct": session_flags.loc[session_flags["viewed"], "carted"].mean() * 100,
    "cart_to_purchase_pct": session_flags.loc[session_flags["carted"], "purchased"].mean() * 100,
    "view_to_purchase_pct": session_flags.loc[session_flags["viewed"], "purchased"].mean() * 100,
}).round(2)

funnel_summary.to_frame("value")


,value
sessions,49271.00
view_sessions,49259.00
cart_sessions,1532.00
purchase_sessions,2672.00
view_to_cart_pct,3.10
cart_to_purchase_pct,53.20
view_to_purchase_pct,5.41


In [14]:
funnel_counts = pd.DataFrame({
    "step": ["Any session", "Viewed", "Carted", "Purchased"],
    "sessions": [
        len(session_flags),
        session_flags["viewed"].sum(),
        session_flags["carted"].sum(),
        session_flags["purchased"].sum(),
    ],
})
funnel_counts["pct_of_all_sessions"] = (funnel_counts["sessions"] / len(session_flags) * 100).round(2)

funnel_counts


,step,sessions,pct_of_all_sessions
0,Any session,49271,100.00
1,Viewed,49259,99.98
2,Carted,1532,3.11
3,Purchased,2672,5.42


## First-Pass Retention Foundations

Retention analysis needs a user-level table. At this stage, we are only checking whether the raw event data contains enough repeat behavior to support cohort analysis and modeling.


In [15]:
user_features = (
    df.groupby("user_id")
      .agg(
          first_event_time=("event_time", "min"),
          last_event_time=("event_time", "max"),
          active_days=("event_time", lambda s: s.dt.date.nunique()),
          sessions=("user_session", "nunique"),
          events=("event_time", "size"),
          products_viewed=("product_id", "nunique"),
          categories_viewed=("category_id", "nunique"),
          cart_events=("event_type", lambda s: (s == "cart").sum()),
          purchase_events=("event_type", lambda s: (s == "purchase").sum()),
          revenue=("price", lambda s: s[df.loc[s.index, "event_type"] == "purchase"].sum()),
      )
      .reset_index()
)

user_features["days_observed"] = (user_features["last_event_time"] - user_features["first_event_time"]).dt.days + 1
user_features["is_buyer"] = user_features["purchase_events"] > 0
user_features["is_repeat_session_user"] = user_features["sessions"] >= 2
user_features["is_multi_day_user"] = user_features["active_days"] >= 2

user_features.head()


,user_id,first_event_time,last_event_time,active_days,sessions,events,products_viewed,categories_viewed,cart_events,purchase_events,revenue,days_observed,is_buyer,is_repeat_session_user,is_multi_day_user
0,275256741,2019-11-01 02:23:03+00:00,2019-11-01 02:23:03+00:00,1,1,1,1,1,0,0,0.00,1,False,False,False
1,295643776,2019-11-01 03:12:38+00:00,2019-11-01 03:15:53+00:00,1,2,8,4,2,0,0,0.00,1,False,True,False
2,306441847,2019-10-01 01:32:09+00:00,2019-10-01 02:56:47+00:00,1,1,2,1,1,0,0,0.00,1,False,False,False
3,356520186,2019-11-01 03:39:27+00:00,2019-11-01 03:49:22+00:00,1,1,6,1,1,0,1,33.45,1,True,False,False
4,362699320,2019-10-01 03:05:49+00:00,2019-10-01 03:25:33+00:00,1,1,7,7,3,0,0,0.00,1,False,False,False


In [16]:
retention_readiness = pd.Series({
    "users": len(user_features),
    "repeat_session_users": user_features["is_repeat_session_user"].sum(),
    "repeat_session_user_pct": user_features["is_repeat_session_user"].mean() * 100,
    "multi_day_users": user_features["is_multi_day_user"].sum(),
    "multi_day_user_pct": user_features["is_multi_day_user"].mean() * 100,
    "buyers": user_features["is_buyer"].sum(),
    "buyer_pct": user_features["is_buyer"].mean() * 100,
}).round(2)

retention_readiness.to_frame("value")


,value
users,40822.00
repeat_session_users,5907.00
repeat_session_user_pct,14.47
multi_day_users,340.00
multi_day_user_pct,0.83
buyers,2484.00
buyer_pct,6.08


In [17]:
user_features[["sessions", "events", "products_viewed", "categories_viewed", "cart_events", "purchase_events", "revenue"]].describe().round(2)


,sessions,events,products_viewed,categories_viewed,cart_events,purchase_events,revenue
count,40822.00,40822.00,40822.00,40822.00,40822.00,40822.00,40822.00
mean,1.21,4.90,3.19,1.43,0.06,0.08,22.12
std,0.76,6.79,4.07,1.13,0.40,0.37,163.70
min,1.00,1.00,1.00,1.00,0.00,0.00,0.00
25%,1.00,1.00,1.00,1.00,0.00,0.00,0.00
50%,1.00,3.00,2.00,1.00,0.00,0.00,0.00
75%,1.00,6.00,4.00,1.00,0.00,0.00,0.00
max,82.00,140.00,76.00,31.00,24.00,26.00,12066.53


## Product, Brand, And Price Checks

These checks identify whether product metadata and pricing look usable for downstream segmentation. This is where a product analyst starts forming hypotheses about assortment, taxonomy, and high-intent categories.


In [18]:
price_summary = df["price"].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).round(2)
price_summary


count   200000.00
mean       284.18
std        351.48
min          0.00
1%           6.18
5%          17.99
25%         64.02
50%        159.59
75%        352.21
95%        979.16
99%       1698.86
max       2574.07
Name: price, dtype: float64

In [19]:
print(f"Rows with price <= 0: {(df['price'] <= 0).sum():,}")

df.loc[df["price"] <= 0, ["event_time", "event_type", "product_id", "category_id", "category_code", "brand", "price"]].head(20)


Rows with price <= 0: 182


,event_time,event_type,product_id,category_id,category_code,brand,price
6258,2019-11-01 00:38:01+00:00,view,33100000,2058719826188173878,<NA>,<NA>,0.00
7245,2019-11-01 00:42:51+00:00,view,33100000,2058719826188173878,<NA>,<NA>,0.00
12743,2019-11-01 01:07:15+00:00,view,12720812,2053013553559896355,<NA>,<NA>,0.00
12908,2019-11-01 01:07:58+00:00,view,12720812,2053013553559896355,<NA>,<NA>,0.00
13503,2019-11-01 01:26:19+00:00,view,38900075,2085718636156158307,<NA>,<NA>,0.00
17220,2019-11-01 01:40:17+00:00,view,13902536,2053013557343158789,construction.components.faucet,<NA>,0.00
18246,2019-11-01 01:43:13+00:00,view,2702683,2053013563911439225,appliances.kitchen.refrigerators,<NA>,0.00
19682,2019-11-01 01:46:58+00:00,view,1307551,2053013558920217191,computers.notebook,<NA>,0.00
20363,2019-11-01 01:48:48+00:00,view,4300485,2053013552385491165,<NA>,<NA>,0.00
20418,2019-11-01 01:48:57+00:00,view,4300485,2053013552385491165,<NA>,<NA>,0.00


In [20]:
top_categories = (
    df.groupby(["category_id", "category_code"], dropna=False)
      .agg(
          events=("event_time", "size"),
          users=("user_id", "nunique"),
          products=("product_id", "nunique"),
          purchase_events=("event_type", lambda s: (s == "purchase").sum()),
          approx_gmv=("price", lambda s: s[df.loc[s.index, "event_type"] == "purchase"].sum()),
      )
      .reset_index()
      .sort_values("events", ascending=False)
)

top_categories.head(20)


,category_id,category_code,events,users,products,purchase_events,approx_gmv
84,2053013555631882655,electronics.smartphone,51666,13287,913,1382,612912.63
38,2053013553559896355,<NA>,10015,2624,2467,174,9671.26
60,2053013554658804075,electronics.audio.headphone,5422,1730,532,145,14664.90
154,2053013558920217191,computers.notebook,5096,1068,594,50,24413.04
52,2053013554415534427,electronics.video.tv,4678,1197,316,85,32463.33
265,2053013563651392361,<NA>,3563,1029,778,46,8741.27
309,2053013565983425517,appliances.environment.vacuum,3475,798,343,52,6945.94
270,2053013563810775923,appliances.kitchen.washer,3436,854,304,75,19896.01
33,2053013553341792533,electronics.clocks,3233,1123,233,49,14920.23
103,2053013557024391671,<NA>,3074,761,157,11,4325.23


In [21]:
top_brands = (
    df.groupby("brand", dropna=False)
      .agg(
          events=("event_time", "size"),
          users=("user_id", "nunique"),
          products=("product_id", "nunique"),
          purchase_events=("event_type", lambda s: (s == "purchase").sum()),
          approx_gmv=("price", lambda s: s[df.loc[s.index, "event_type"] == "purchase"].sum()),
      )
      .reset_index()
      .sort_values("events", ascending=False)
)

top_brands.head(20)


,brand,events,users,products,purchase_events,approx_gmv
1990,<NA>,30169,9495,7685,262,37781.49
1572,samsung,23431,7622,617,726,187007.82
107,apple,18186,6663,360,547,412909.92
1947,xiaomi,14358,4256,452,260,38496.80
847,huawei,4929,1746,101,105,23196.34
1086,lucente,3134,1142,334,50,14602.40
276,bosch,2635,967,471,24,7914.44
1671,sony,2109,809,356,22,10102.75
1063,lg,2100,857,201,30,11020.32
1339,oppo,2035,734,34,40,9767.69


## Full-Data QA Helper

The notebook uses a sample by default. Use this chunked helper for full-data checks that should not depend on the sample, such as taxonomy consistency.


In [22]:
def check_category_code_consistency_full(raw_files, chunksize=500_000):
    has_missing = set()
    has_known = set()

    for file_path in raw_files:
        for chunk in pd.read_csv(file_path, usecols=["category_id", "category_code"], chunksize=chunksize):
            category_id = chunk["category_id"].astype("string")
            category_code = chunk["category_code"].astype("string")
            is_missing = category_code.isna() | category_code.str.strip().eq("")

            has_missing.update(category_id[is_missing].dropna().unique())
            has_known.update(category_id[~is_missing].dropna().unique())

    return sorted(has_missing & has_known)

# Uncomment when you want the full raw-data check. This can take a few minutes.
# category_ids_missing_and_known_full = check_category_code_consistency_full(RAW_FILES)
# len(category_ids_missing_and_known_full), category_ids_missing_and_known_full[:20]


## EDA Takeaways And Handoff

Update these bullets after running the notebook top-to-bottom.

**What we know from data exploration**

- The dataset supports event-level, session-level, user-level, product-level, category-level, and brand-level cuts.
- Funnel analysis should use session and user denominators, not only event counts.
- Retention analysis should define return behavior explicitly: repeat session, multi-day return, repeat purchase, or cohort return.
- Category taxonomy missingness needs to be handled before category-level recommendations.
- Purchase rows are product-level events, so order-level metrics require caution.

**Handoff to next notebooks**

- `02_behavioral_eda.ipynb`: compare buyers vs non-buyers, repeat vs one-time users, and high vs low engagement sessions.
- `03_funnel_analysis.ipynb`: quantify drop-off from view to cart to purchase, then segment by category, brand, and price band.
- `04_retention_analysis.ipynb`: define retained users and run cohort views by first activity date or first purchase date.
- `05_retention_modeling.ipynb`: build user/session features from early behavior and predict retention.
